# 12 · Cross-validation y optimización de hiperparámetros

La validación no es un detalle administrativo: define qué significa 'generalizar'. Un split incorrecto puede producir resultados impecables en notebook y desastrosos en producción.

## Objetivos
- Diferenciar parámetros e hiperparámetros.
- Usar KFold, StratifiedKFold, GroupKFold y TimeSeriesSplit.
- Comparar Grid Search, Random Search y optimización bayesiana.
- Aplicar nested CV para evaluación honesta.
- Evitar leakage al hacer preprocessing y feature selection.
- Entender multiple-comparisons / overfitting al validation set.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import *
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
SEED=42
X,y=load_breast_cancer(return_X_y=True)

## 1. ¿Qué split representa el futuro?
- `KFold`: observaciones iid, regresión o clases suficientemente mezcladas.
- `StratifiedKFold`: conserva proporciones de clases.
- `GroupKFold`: una entidad no puede aparecer en train y validation. Ej.: persona, hogar, paciente, institución, dispositivo.
- `TimeSeriesSplit`: respeta orden temporal.
- splits geográficos: para medir transferencia territorial.

Si múltiples registros de la misma persona caen en train y test, el modelo puede memorizar identidad/contexto y la métrica queda inflada.


In [ ]:
# Visualización simple de folds
fig,axes=plt.subplots(3,1,figsize=(12,5),sharex=True)
for ax,(name,splitter) in zip(axes,[('KFold',KFold(5,shuffle=True,random_state=SEED)),('StratifiedKFold',StratifiedKFold(5,shuffle=True,random_state=SEED)),('TimeSeriesSplit',TimeSeriesSplit(5))]):
 for i,(tr,va) in enumerate(splitter.split(X,y)):
  ax.scatter(tr,np.full_like(tr,i),s=3); ax.scatter(va,np.full_like(va,i),s=6)
 ax.set_title(name)
plt.tight_layout(); plt.show()

## 2. Grid vs Random Search
Grid explora todas las combinaciones definidas. Si solo unos pocos hiperparámetros importan, Random Search suele cubrir más valores útiles con el mismo presupuesto. Para parámetros en órdenes de magnitud (`C`, learning rate), muestrea en escala logarítmica.


In [ ]:
pipe=Pipeline([('scale',StandardScaler()),('svm',SVC(probability=True))])
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
grid=GridSearchCV(pipe,{'svm__C':[.01,.1,1,10,100],'svm__gamma':['scale',.001,.01,.1,1],'svm__kernel':['rbf']},cv=cv,scoring='roc_auc',n_jobs=-1)
random=RandomizedSearchCV(pipe,{'svm__C':np.logspace(-3,3,500),'svm__gamma':np.logspace(-5,1,500),'svm__kernel':['rbf']},n_iter=30,cv=cv,scoring='roc_auc',random_state=SEED,n_jobs=-1)
for s in [grid,random]: s.fit(X,y); print(type(s).__name__,s.best_score_,s.best_params_)

## 3. Optimización bayesiana con Optuna
Optuna usa resultados anteriores para decidir qué probar después y permite pruning de trials poco prometedores. Es especialmente útil cuando entrenar es caro. No elimina la necesidad de CV ni convierte hiperparámetros en ciencia exacta.


In [ ]:
!pip -q install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
 C=trial.suggest_float('C',1e-3,1e3,log=True); gamma=trial.suggest_float('gamma',1e-5,1,log=True)
 m=Pipeline([('scale',StandardScaler()),('svm',SVC(C=C,gamma=gamma))])
 return cross_val_score(m,X,y,cv=cv,scoring='roc_auc',n_jobs=-1).mean()
study=optuna.create_study(direction='maximize'); study.optimize(objective,n_trials=25)
print(study.best_value,study.best_params)

## 4. Nested Cross-Validation
Si usamos los mismos folds para escoger hiperparámetros y reportar performance, la estimación se vuelve optimista. Nested CV:
1. outer fold separa una evaluación limpia;
2. inner folds seleccionan hiperparámetros;
3. se reportan scores de outer folds.


In [ ]:
outer=StratifiedKFold(5,shuffle=True,random_state=1); inner=StratifiedKFold(4,shuffle=True,random_state=2)
search=RandomizedSearchCV(pipe,{'svm__C':np.logspace(-3,3,200),'svm__gamma':np.logspace(-5,1,200)},n_iter=15,cv=inner,scoring='roc_auc',random_state=SEED,n_jobs=-1)
nested=cross_val_score(search,X,y,cv=outer,scoring='roc_auc',n_jobs=-1)
print('nested scores',nested,'mean=',nested.mean(),'std=',nested.std())

## 5. Overfitting al validation set
Si pruebas 10.000 configuraciones, alguna parecerá excepcional por azar. Repetir experimentos, usar nested CV, mantener un holdout final y limitar el espacio con conocimiento previo reduce este problema.

## 6. Learning curves
Sirven para decidir si más datos podrían ayudar. Si train y validation son bajos y cercanos: alto bias. Si train alto y validation mucho menor: variance.


In [ ]:
from sklearn.model_selection import learning_curve
train_sizes,tr,va=learning_curve(Pipeline([('scale',StandardScaler()),('svm',SVC(C=2))]),X,y,cv=cv,scoring='roc_auc',train_sizes=np.linspace(.1,1,8),n_jobs=-1)
plt.plot(train_sizes,tr.mean(1),label='train'); plt.plot(train_sizes,va.mean(1),label='validation'); plt.fill_between(train_sizes,va.mean(1)-va.std(1),va.mean(1)+va.std(1),alpha=.2); plt.legend(); plt.xlabel('n train'); plt.ylabel('ROC-AUC'); plt.show()

## Errores comunes
- CV aleatoria para series de tiempo;
- paciente/hogar en ambos lados;
- preprocessing antes de CV;
- reportar `best_score_` como test performance;
- tunear demasiado sobre un dataset pequeño;
- no guardar semillas ni folds;
- optimizar una métrica que no representa el uso.

## Ejercicios
1. Crea IDs de grupo y demuestra inflación con KFold vs GroupKFold.
2. Implementa RepeatedStratifiedKFold y compara incertidumbre.
3. Compara Grid vs Random con el mismo número de evaluaciones.
4. Usa Optuna para Random Forest o XGBoost.
5. Implementa nested CV y reporta intervalo de confianza.
6. Crea una validación temporal rolling-window.
7. Diseña splits para un sistema nacional con años, regiones y personas.
